# Ordenación cronológica y partición temporal en Train / Val / Test

Se aplica un split estrictamente cronológico (70/15/15) para simular despliegue prospectivo. Este enfoque es requerido por TRIPOD-AI y previene la fuga de información temporal.

## Configuración y librerías

Importación de dependencias y definición de rutas de entrada (`data/interim/`) y salida (`data/processed/`).

In [1]:
import os
import json
import pandas as pd
from pathlib import Path

In [2]:
# Rutas relativas al notebook ejecutado desde notebooks/01_eda/
INTERIM_DIR = Path("../../data/interim")
PROCESSED_DIR = Path("../../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio de lectura: {INTERIM_DIR.resolve()}")
print(f"Directorio de escritura: {PROCESSED_DIR.resolve()}")

Directorio de lectura: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\interim
Directorio de escritura: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\processed


## Carga de la cohorte con etiquetas

Se lee el Parquet generado por el notebook `03_verify_build_labels` que contiene la cohorte filtrada con las etiquetas L1, L2 y L3.

In [3]:
# Carga de la cohorte generada en el notebook 03
file_path = INTERIM_DIR / "cohort_with_labels.parquet"

print("Cargando datos...")
df_cohort = pd.read_parquet(file_path)

print(f"Total de estancias en la cohorte: {len(df_cohort):,}")

Cargando datos...
Total de estancias en la cohorte: 397,601


## Ordenación cronológica

La cohorte se ordena por `intime` antes de la partición para garantizar que los cortes de índice sean equivalentes a cortes temporales.

In [4]:
df_cohort['intime'] = pd.to_datetime(df_cohort['intime'])
df_cohort = df_cohort.sort_values('intime').reset_index(drop=True)

print(f"Primer paciente ingresado: {df_cohort['intime'].min()}")
print(f"Último paciente ingresado: {df_cohort['intime'].max()}")

Primer paciente ingresado: 2110-01-11 03:43:00
Último paciente ingresado: 2212-04-05 23:23:00


## Partición en conjuntos Train / Val / Test

In [5]:
# Índices de corte calculados sobre el total de filas ordenadas cronológicamente
train_ratio = 0.70
val_ratio = 0.15

n_total = len(df_cohort)
train_end = int(n_total * train_ratio)
val_end = train_end + int(n_total * val_ratio)

# La ordenación previa asegura que los cortes por índice son temporales
df_train = df_cohort.iloc[:train_end].copy()
df_val   = df_cohort.iloc[train_end:val_end].copy()
df_test  = df_cohort.iloc[val_end:].copy()

print("=== Tamaños del Split ===")
print(f"Train: {len(df_train):,} filas ({len(df_train)/n_total*100:.2f}%)")
print(f"Val:   {len(df_val):,} filas ({len(df_val)/n_total*100:.2f}%)")
print(f"Test:  {len(df_test):,} filas ({len(df_test)/n_total*100:.2f}%)")

=== Tamaños del Split ===
Train: 278,320 filas (70.00%)
Val:   59,640 filas (15.00%)
Test:  59,641 filas (15.00%)


## Verificación de ausencia de data leakage temporal

Se comprueba que el máximo `intime` de train es anterior al mínimo de val, y que el máximo de val es anterior al mínimo de test. Un fallo en los asserts indicaría fuga de datos.

In [6]:
max_train = df_train['intime'].max()
min_val   = df_val['intime'].min()
max_val   = df_val['intime'].max()
min_test  = df_test['intime'].min()

print("=== Límites Temporales ===")
print(f"Fin de Train:   {max_train}")
print(f"Inicio de Val:  {min_val}")
print(f"Fin de Val:     {max_val}")
print(f"Inicio de Test: {min_test}")

# Si hay fuga temporal, los asserts fallan con un error explícito
assert max_train <= min_val, "ERROR: Fuga de datos detectada entre Train y Val"
assert max_val <= min_test, "ERROR: Fuga de datos detectada entre Val y Test"

print("\nVerificacion superada: split temporal estrictamente cronologico.")

=== Límites Temporales ===
Fin de Train:   2172-08-16 00:53:00
Inicio de Val:  2172-08-16 04:30:00
Fin de Val:     2184-06-22 21:05:00
Inicio de Test: 2184-06-22 21:52:00

Verificacion superada: split temporal estrictamente cronologico.


## Exportación a `data/processed/`

Se guardan los tres conjuntos en Parquet junto con un JSON de metadatos del split para trazabilidad y reproducibilidad del experimento.

In [7]:
df_train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
df_val.to_parquet(PROCESSED_DIR / "val.parquet", index=False)
df_test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

# Metadatos del split para trazabilidad del experimento
split_metadata = {
    "total_rows": len(df_cohort),
    "train": {"rows": len(df_train), "start_date": str(df_train['intime'].min()), "end_date": str(max_train)},
    "val": {"rows": len(df_val), "start_date": str(min_val), "end_date": str(max_val)},
    "test": {"rows": len(df_test), "start_date": str(min_test), "end_date": str(df_test['intime'].max())}
}

with open(PROCESSED_DIR / "splits.json", "w") as f:
    json.dump(split_metadata, f, indent=4)

print(f"Archivos guardados en: {PROCESSED_DIR.resolve()}")

Archivos guardados en: C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data\processed
